# 7. Bonus: uncertainty that pays for itself

If the model can say where it is uncertain, it should be able to say which
measurement to take next. Start from 20 labels, repeatedly label the points the
model is least sure about, and compare against labelling at random.

You write `select_indices`. About 4 hours including the write-up; the experiment
itself runs in a couple of minutes.

Be ready for the answer to be that random wins. On `toy1d` it does, reproducibly,
and explaining why is a better report than a narrow win would have been.

## 7.1 What to acquire on

The decision-theoretic answer is to label the point that most reduces uncertainty
about the model. That is the mutual information between the label and the weights,
and mutual information is symmetric, so

    I(W; y | x) = H[ E_W p(y|x,W) ] - E_W[ H(p(y|x,W)) ]

which is exactly the epistemic term you already compute. That identity is the
reason the decomposition in notebook 02 was worth implementing: the quantity you
need is not computable in its first form, because it involves the posterior after a
label you do not have, but it is computable in its second.

**Acquire on the epistemic part, never on the total.** Aleatoric uncertainty is
irreducible, so a point that is uncertain only because it is noisy teaches the
model nothing while costing a full label. If uncertainty sampling appears to win
easily here, check whether you scored on the total by mistake.

And always run the random baseline. An active learning result without one is not a
result.

In [ ]:
import sys
import time
from dataclasses import replace

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import torch

from bdl.data import corrupt, load_track
from bdl.metrics import auroc, interval_coverage, predictive_nll
from bdl.models import build_model, default_loss, enable_dropout, fit, gaussian_head, set_seed
from bdl.plots import plot_learning_curves, plot_shift_sweep
from bdl.store import run_dir

%matplotlib inline

TRACK = "toy1d"  # toy1d is the interesting case; your own track also works


# The functions you wrote in notebooks 01 and 02, repeated so this notebook stands
# on its own.
@torch.no_grad()
def predict_mc_dropout(model, x, n_samples=30):
    model.eval()
    enable_dropout(model)
    mus, sigmas = [], []
    for _ in range(n_samples):
        mu, sigma = gaussian_head(model(x))
        mus.append(mu)
        sigmas.append(sigma)
    return torch.stack(mus), torch.stack(sigmas)


def decompose_variance(mu, sigma):
    aleatoric = (sigma**2).mean(dim=0)
    epistemic = mu.var(dim=0, unbiased=False)
    return aleatoric + epistemic, aleatoric, epistemic


def calibration_error(mu, sigma, y, n_bins=15):
    levels = torch.linspace(0.0, 1.0, n_bins + 2)[1:-1]
    return float(np.mean([abs(interval_coverage(mu, sigma, y, float(q)) - float(q)) for q in levels]))

## 7.2 The acquisition function

Two strategies:

* `"random"`: draw `n_acquire` indices uniformly without replacement. This is the
  baseline that has to be beaten.
* `"uncertainty"`: predict on the pool, score every pool point by its epistemic
  variance, and take the `n_acquire` highest.

In [ ]:
def select_indices(strategy, model, x_pool, n_acquire, rng, n_samples=30):
    """Choose which pool points to label next. Returns indices into x_pool."""
    # ---- TODO ------------------------------------------------------------
    # "random"      : rng.choice over len(x_pool), without replacement
    # "uncertainty" : predict on x_pool, score by the EPISTEMIC part of
    #                 decompose_variance, and take the n_acquire highest
    raise NotImplementedError
    # ----------------------------------------------------------------------

## 7.3 The loop

Provided. One run trains a fresh model from scratch at every round, which is the
honest way to do this: reusing weights across rounds would confound the acquisition
strategy with a warm start.

In [ ]:
def run_active_learning(track, strategy, seed, n_initial=20, n_acquire=10, n_rounds=10, epochs=300):
    """One active-learning run. Returns (labelled counts, test NLL at each step)."""
    ds = load_track(track)
    rng = np.random.default_rng(seed)

    labelled = rng.choice(len(ds.x_train), size=n_initial, replace=False)
    pool = np.setdiff1d(np.arange(len(ds.x_train)), labelled)

    counts, nlls = [], []
    for _ in range(n_rounds + 1):
        sub = replace(ds, x_train=ds.x_train[labelled], y_train=ds.y_train[labelled])
        set_seed(seed)
        model = build_model(sub, hidden=(64, 64), dropout=0.1)
        fit(model, sub.x_train, sub.y_train, loss_fn=default_loss(sub), epochs=epochs, lr=1e-2, seed=seed)

        mu, sigma = predict_mc_dropout(model, ds.x_test, n_samples=30)
        counts.append(len(labelled))
        nlls.append(predictive_nll(mu, sigma, ds.y_test))

        if len(pool) == 0:
            break
        picked = select_indices(strategy, model, ds.x_train[pool], n_acquire, rng)
        labelled = np.concatenate([labelled, pool[picked]])
        pool = np.delete(pool, picked)

    return counts, nlls

In [ ]:
# ---- check your work -------------------------------------------------------
ds_check = load_track(TRACK)
set_seed(0)
model_check = build_model(ds_check, hidden=(16, 16), dropout=0.1)
fit(model_check, ds_check.x_train, ds_check.y_train, loss_fn=default_loss(ds_check), epochs=50, seed=0)
pool_check = ds_check.x_train[:60]
rng_check = np.random.default_rng(0)

for strategy in ("random", "uncertainty"):
    idx = select_indices(strategy, model_check, pool_check, 10, rng_check)
    idx = np.asarray(idx)
    assert len(idx) == 10, (strategy, len(idx))
    assert len(set(idx.tolist())) == 10, f"{strategy} returned duplicate indices"
    assert idx.min() >= 0 and idx.max() < len(pool_check), f"{strategy} returned out-of-range indices"

# The uncertainty strategy must actually be selecting the most uncertain points.
mu_c, sd_c = predict_mc_dropout(model_check, pool_check, n_samples=30)
_, _, epi_c = decompose_variance(mu_c, sd_c)
chosen = np.asarray(select_indices("uncertainty", model_check, pool_check, 10, rng_check))
assert float(epi_c[chosen].mean()) > float(epi_c.mean()), (
    "the chosen points are not more uncertain than average: are you scoring on "
    "epistemic uncertainty, and taking the highest rather than the lowest?"
)
print("OK   both strategies return valid, distinct indices, and uncertainty picks uncertain points")

## 7.4 Five seeds, two strategies

Never report a single-seed learning curve. The difference between acquisition
strategies is routinely smaller than the difference between seeds, so a single run
can show whatever you hoped it would.

In [ ]:
SEEDS = 5
curves = {}
finals = {}

t0 = time.perf_counter()
for strategy in ("random", "uncertainty"):
    runs = []
    for seed in range(SEEDS):
        counts, nlls = run_active_learning(TRACK, strategy, seed)
        runs.append(nlls)
        print(f"  {strategy:12s} seed {seed}: final NLL {nlls[-1]:.4f}")
    arr = np.array(runs)
    curves[strategy] = (np.array(counts), arr.mean(0), arr.std(0))
    finals[strategy] = arr[:, -1]
print(f"\n{2 * SEEDS} runs in {time.perf_counter() - t0:.0f}s")

In [ ]:
plot_learning_curves(
    curves,
    run_dir("bonus_al") / "active_learning.png",
    title=f"Active learning on {TRACK}, MC Dropout, {SEEDS} seeds",
);

In [ ]:
gap = float(finals["random"].mean() - finals["uncertainty"].mean())
spread = float(max(finals["random"].std(), finals["uncertainty"].std()))
paired = finals["random"] - finals["uncertainty"]

print(f"final NLL   random      {finals['random'].mean():.4f} +/- {finals['random'].std():.4f}")
print(f"            uncertainty {finals['uncertainty'].mean():.4f} +/- {finals['uncertainty'].std():.4f}")
print(f"difference (random - uncertainty) {gap:+.4f}, seed-to-seed spread {spread:.4f}")
print(f"paired per-seed differences: {[f'{d:+.3f}' for d in paired]}")

if abs(gap) < spread:
    print(
        "\nThe gap is smaller than the spread across seeds. On this evidence you "
        "cannot claim\neither strategy is better, and the correct conclusion is that "
        "this experiment could\nnot detect a difference. Write that: it is graded as "
        "a result."
    )
else:
    better = "random" if gap < 0 else "uncertainty"
    print(f"\nThe gap exceeds the seed spread, so it is a real effect: {better} acquisition wins.")

## 7.5 Why random wins here

In the reference run random acquisition ends around 0.24 nats better than
uncertainty sampling, against a seed spread of about 0.15, so the effect is real.
Three mechanisms, in decreasing order of size.

**The two uncertainties peak in the same place.** `toy1d`'s observation noise grows
with `x`, and the epistemic uncertainty is largest at the edges of the input range.
Scoring on epistemic variance sends the budget to the right-hand edge, which is
both unfamiliar and irreducibly noisy, and after the first couple of rounds the
noise is what dominates.

**The scale-free version of the score would partly fix this.** For a Gaussian
predictive with epistemic variance `v` and noise `sigma^2`, the mutual information
in section 7.1 works out to

    I = 0.5 * log(1 + v / sigma^2)

which discounts epistemic variance by the local noise floor, whereas raw `v` does
not. Swapping one for the other, one variable at a time, five seeds, is a good
experiment and a good candidate for the open-ended part of the project.

**Greedy batches are redundant.** Labels are conditionally independent given the
weights, so the information in a batch is at most the sum of the information in its
members, and strictly less when they are informative about the same weights. Taking
the ten highest-scoring points takes ten near-duplicates from the same edge. Random
acquisition is accidentally diverse.

## 7.6 Alternative: sweep the shift instead

If you would rather measure where error bars stop meaning anything, sweep the
corruption strength on the test inputs and watch NLL, calibration and OOD detection
as it grows. `corrupt` adds Gaussian noise measured in training standard
deviations, so the knob means the same thing on every track.

This is a different bonus, not an addition to the one above. Either is enough.

In [ ]:
ds_sweep = load_track(TRACK)
strengths = [0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0]
sweep = {}

for name, dropout_p in (("mc_dropout", 0.1),):
    set_seed(0)
    model_s = build_model(ds_sweep, hidden=(64, 64), dropout=dropout_p)
    fit(
        model_s, ds_sweep.x_train, ds_sweep.y_train, loss_fn=default_loss(ds_sweep),
        epochs=400, lr=1e-2, seed=0,
    )
    clean_mu, clean_sd = predict_mc_dropout(model_s, ds_sweep.x_test, n_samples=30)
    _, _, clean_epi = decompose_variance(clean_mu, clean_sd)

    series = {"nll": [], "ece": [], "ood_auroc": []}
    for s in strengths:
        mu_s, sd_s = predict_mc_dropout(
            model_s, corrupt(ds_sweep.x_test, s, seed=1), n_samples=30
        )
        _, _, epi_s = decompose_variance(mu_s, sd_s)
        series["nll"].append(predictive_nll(mu_s, sd_s, ds_sweep.y_test))
        series["ece"].append(calibration_error(mu_s, sd_s, ds_sweep.y_test))
        series["ood_auroc"].append(auroc(clean_epi, epi_s))
    sweep[name] = series
    print(f"{name}: NLL {series['nll'][0]:.3f} -> {series['nll'][-1]:.3f}, "
          f"ECE {series['ece'][0]:.3f} -> {series['ece'][-1]:.3f}")

plot_shift_sweep(sweep, strengths, run_dir("bonus_al") / "shift_sweep.png");

## 7.7 What to report

* the learning curves with error bands across at least five seeds, and the random
  baseline;
* the gap between strategies next to the seed-to-seed spread, and the conclusion
  that comparison supports, including "this experiment could not detect a
  difference" if that is what it shows;
* the mechanism, not just the outcome. Look at *where* in the input range the
  acquired points ended up;
* for the shift sweep instead: the strength at which each metric stops being
  usable, stated as a number.